# Session 6 — Multi-Case Automation and Reproducible CFD Reports

**Post-CFD Analysis with Python | Dr. Nuha Aljuneidi**

Real projects rarely involve one CFD run — they involve a sweep of cases (Reynolds number, angle of attack, design variants). This session automates the Session 1–5 workflow across a case sweep and produces a reproducible, auditable summary report.

## Learning outcomes
- Structure a parameter sweep as reusable functions, not copy-pasted cells.
- Run automated data-quality checks across every case in a sweep.
- Tabulate and compare results across cases.
- Produce a reproducible report with explicit pass/fail quality flags.

## Using your own Fluent or CSV data
This notebook uses synthetic data so you can run every cell immediately without a CFD license. When you are ready to use your own results, export a CSV from Fluent (or any solver) with coordinates, variable names, units, operating conditions, and a case identifier, then replace the synthetic-data cell below with:

```python
df = pd.read_csv("your_export.csv")
```

Map your solver's column names to the ones used in this notebook before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import hashlib
from datetime import datetime

rng = np.random.default_rng(41)
print("Environment ready.")

## 1. Defining the sweep

Reproducibility starts with parameters defined once, in one place — never re-typed inside a loop or hidden inside a plot call. The sweep below varies Reynolds number and angle of attack as a **full factorial grid** (every Reynolds number paired with every angle), not as a single list where both change together. That matters: if Re and alpha changed together case-by-case, you could never tell which variable caused a change in a result — the two effects would be confounded. With a full grid, each variable's effect can be read off while holding the other fixed.

In [ ]:
reynolds_numbers = [5.0e4, 5.0e5]
angles_of_attack_deg = [0.0, 4.0, 8.0, 12.0]
re_labels = {5.0e4: "Re_lo", 5.0e5: "Re_hi"}

sweep_cases = pd.DataFrame(
    [{"case_id": f"{re_labels[re_]}_a{a:g}deg", "reynolds_number": re_, "angle_of_attack_deg": a}
     for re_, a in itertools.product(reynolds_numbers, angles_of_attack_deg)]
)
U_inf_mps = 20.0
rho_kgpm3 = 1.225
chord_m = 0.15
sweep_cases

## 2. A reusable case-processing function

Wrap the Session 1 quality checks and a synthetic lift/drag calculation into one function so every case is processed identically — this is what makes the sweep reproducible.

In [ ]:
def run_synthetic_case(reynolds_number, alpha_deg, rng):
    """Return a synthetic (Cl, Cd, quality_flags) for one case, mimicking post-processed CFD output."""
    alpha_rad = np.deg2rad(alpha_deg)
    Cl = 2 * np.pi * alpha_rad * (1 - 0.05 * np.log10(reynolds_number / 1e5).clip(min=0)) + rng.normal(0, 0.01)

    # Laminar flat-plate skin-friction trend (Blasius): Cf ~ 1.328 / sqrt(Re) -- a simplified,
    # physically-grounded stand-in for Re-dependent profile drag. Unlike a near-flat function of Re,
    # this actually varies by a reportable amount across the sweep, and drops below the 0.005 floor
    # at high Re -- so the floor is doing real work for some cases, not none of them.
    Cd0 = 1.328 / np.sqrt(reynolds_number)
    Cd = max(Cd0, 0.005) + 0.02 * alpha_rad ** 2 + rng.normal(0, 0.0005)

    residual_final = 10 ** rng.uniform(-8, -5)
    force_drift_pct = rng.uniform(0.1, 2.0)

    quality_flags = {
        "residual_below_1e-6": residual_final < 1e-6,
        "force_drift_below_1pct": force_drift_pct < 1.0,
        "Cl_within_physical_range": -0.5 < Cl < 2.0,
        "Cd_within_physical_range": 0.0 < Cd < 0.5,
    }
    return Cl, Cd, quality_flags, residual_final, force_drift_pct

# Smoke-test on one case before running the full sweep
Cl, Cd, flags, resid, drift = run_synthetic_case(1e5, 4.0, rng)
print(f"Cl={Cl:.4f}, Cd={Cd:.4f}, flags={flags}")

### Checkpoint 1 — Read the function, don't just run it
Before proceeding: using the `Cd0 = 1.328 / sqrt(Re)` trend now in `run_synthetic_case`, compute `Cd0` at `Re = 6e5` and at `Re = 5e4` by hand (or in a scratch cell). At which of those two is the `max(Cd0, 0.005)` floor actually changing the result, and at which is it not? Explaining a function you didn't write — including when a safety floor like this actually engages — is part of using automation responsibly.

## 3. Running the sweep with automated quality checks

Apply `run_synthetic_case` to every row of `sweep_cases`, collecting results and quality flags into one results table — no case is hand-processed differently from another.

In [ ]:
results = []
for _, row in sweep_cases.iterrows():
    Cl, Cd, flags, resid, drift = run_synthetic_case(row.reynolds_number, row.angle_of_attack_deg, rng)
    all_pass = all(flags.values())
    results.append({
        "case_id": row.case_id,
        "reynolds_number": row.reynolds_number,
        "angle_of_attack_deg": row.angle_of_attack_deg,
        "Cl": Cl, "Cd": Cd,
        "residual_final": resid,
        "force_drift_pct": drift,
        "quality_pass": all_pass,
        **{f"flag_{k}": v for k, v in flags.items()},
    })

results_df = pd.DataFrame(results)
results_df

### Checkpoint 2 — Find the failing case
At least one case above is likely to fail a quality flag (residual or force-drift thresholds use random draws each run). Filter `results_df` to show only rows where `quality_pass` is `False`, and identify which specific flag(s) failed.

In [ ]:
# TODO: filter results_df to rows where quality_pass is False and print which flag_* columns are False


## 4. Comparing cases

With every case processed identically — and Reynolds number and angle of attack varied independently — you can compare cases two ways: lift and drag vs. angle of attack *at a fixed Reynolds number*, and the effect of Reynolds number alone *at a fixed angle of attack*. Confounded data (both variables changing together) cannot support either comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for re_val, group in results_df.groupby("reynolds_number"):
    label = f"Re = {re_val:.1e}"
    axes[0].plot(group.angle_of_attack_deg, group.Cl, marker="o", label=label)
    axes[1].plot(group.angle_of_attack_deg, group.Cd, marker="o", label=label)

axes[0].set_xlabel("angle of attack (deg)")
axes[0].set_ylabel("Cl (-)")
axes[0].set_title("Lift vs. angle of attack, by Reynolds number")
axes[0].legend()

axes[1].set_xlabel("angle of attack (deg)")
axes[1].set_ylabel("Cd (-)")
axes[1].set_title("Drag vs. angle of attack, by Reynolds number")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Cl/Cd ratio by case:")
for _, r in results_df.iterrows():
    print(f"  {r.case_id}: L/D = {r.Cl/r.Cd:6.2f}  (Re={r.reynolds_number:.1e}, alpha={r.angle_of_attack_deg:.1f} deg)")

## 5. A reproducible report

A reproducible report states exactly what was run, when, with what parameters, and which cases passed quality control — so anyone can regenerate or audit it.

In [ ]:
def build_run_metadata(results_df, rng_seed):
    """Collect the fields a real reproducibility record needs, beyond just a random seed."""
    case_ids = results_df["case_id"].tolist()
    case_fingerprint = hashlib.sha256(",".join(case_ids).encode()).hexdigest()[:12]
    return {
        "rng_seed": rng_seed,
        "case_ids": case_ids,
        "case_fingerprint": case_fingerprint,
        "solver_version": "Fluent 2024 R2 (example — replace with your solver's reported version)",
        "mesh_identifier": "mesh_v3_wallYplus_lt1 (example — replace with your mesh name/ID)",
        "code_version": "session-06-student.ipynb v1.1",
        "processing_timestamp": datetime.now().isoformat(timespec="seconds"),
    }

def generate_report(results_df, sweep_params, run_metadata):
    lines = []
    lines.append(f"CFD Sweep Report — generated {run_metadata['processing_timestamp']}")
    lines.append(f"Reference conditions: U_inf = {sweep_params['U_inf_mps']} m/s, "
                 f"rho = {sweep_params['rho_kgpm3']} kg/m^3, chord = {sweep_params['chord_m']} m")
    lines.append(f"Cases run: {len(results_df)}   "
                 f"Passed quality control: {results_df['quality_pass'].sum()} / {len(results_df)}")
    lines.append("")
    lines.append("Reproducibility record:")
    lines.append(f"  rng seed: {run_metadata['rng_seed']}")
    lines.append(f"  case IDs: {', '.join(run_metadata['case_ids'])}")
    lines.append(f"  case fingerprint (sha256[:12] of case IDs): {run_metadata['case_fingerprint']}")
    lines.append(f"  solver version: {run_metadata['solver_version']}")
    lines.append(f"  mesh identifier: {run_metadata['mesh_identifier']}")
    lines.append(f"  code version: {run_metadata['code_version']}")
    lines.append("")
    for _, r in results_df.iterrows():
        status = "PASS" if r.quality_pass else "FAIL"
        lines.append(f"[{status}] {r.case_id}: Re={r.reynolds_number:.2e}, "
                     f"alpha={r.angle_of_attack_deg:.1f} deg -> "
                     f"Cl={r.Cl:.4f}, Cd={r.Cd:.4f}, L/D={r.Cl/r.Cd:.2f}")
        if not r.quality_pass:
            failed = [k.replace("flag_", "") for k in results_df.columns
                      if k.startswith("flag_") and not r[k]]
            lines.append(f"         failed checks: {', '.join(failed)}")
    return "\n".join(lines)

run_metadata = build_run_metadata(results_df, rng_seed=41)
report_text = generate_report(results_df, {"U_inf_mps": U_inf_mps, "rho_kgpm3": rho_kgpm3, "chord_m": chord_m}, run_metadata)
print(report_text)

### Checkpoint 3 — Detect changes to the results themselves
The report now records a random seed, the exact case IDs run, and placeholder solver/mesh/code-version fields — but the `case_fingerprint` only proves *which cases* ran, not that their *numeric results* weren't altered afterward. Extend `build_run_metadata` (or write a small addition) to also hash the actual results content — for example, `hashlib.sha256(results_df.to_csv(index=False).encode()).hexdigest()[:12]` — and add it to the printed report as a `results_fingerprint`. Two runs with identical inputs should now be verifiable as identical outputs, not just identical case lists.

In [ ]:
# TODO: compute a results_fingerprint by hashing results_df.to_csv(index=False), e.g.:
#   results_fingerprint = hashlib.sha256(results_df.to_csv(index=False).encode()).hexdigest()[:12]
# then print it alongside (or re-generate) the report so it includes this field.


## Graduate/Advanced Extension
The main sweep above already uses a factorial Reynolds-number x angle-of-attack grid (2 x 4 = 8 cases). Extend it to a much finer grid — e.g., 5 Reynolds numbers x 6 angles = 30 cases, still via `itertools.product` — re-run `run_synthetic_case` over all combinations, and produce a heatmap of `Cl/Cd` over the (Re, alpha) grid using `ax.pcolormesh` or `ax.contourf`. Overlay a marker on any cell whose case failed a quality flag, so the heatmap shows both the aerodynamic trend and where the quality-controlled results actually came from. This is the natural end point of the course: from a single trustworthy CFD export (Session 1) to an automated, quality-checked, statistically separable design-space study (Session 6).

## Exit ticket
In three sentences: name one part of your own post-processing workflow you would now turn into a reusable function instead of copy-pasted cells, state why an automated quality flag is more trustworthy than a manual "eyeball check" across many cases, and describe what "reproducible" means for a CFD report in your own words.

**Course complete.** You now have a working pipeline from raw CFD export to trustworthy, automated, reproducible engineering results.